# Domain 3 — Race Strategy EDA

**Notebook 06 | Domain 3: Race Strategy & Pit-Stop Optimisation**

This notebook explores the cross-domain (XT) strategy features generated by
`src/features/build_xt_features.py` and performs exploratory data analysis to
understand:

1. Distribution of pit-urgency scores across laps and compounds
2. Undercut opportunity patterns by circuit and lap
3. Safety-car delta features and their relationship to track status
4. Composite strategy score distributions and recommended actions

## Prerequisites

```bash
# Ensure features_xt.db has been populated
python -m src.features.build_xt_features --data-dir data/
```


In [ ]:
import sqlite3
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="Set2")

DB_XT   = Path("../data/features_xt.db")
FIG_DIR = Path("../docs/figures/domain3_strategy")
FIG_DIR.mkdir(parents=True, exist_ok=True)

print("Libraries loaded ✓")


In [ ]:
def load_table(db: Path, table: str) -> pd.DataFrame:
    if not db.exists():
        print(f"⚠ {db} not found — using synthetic demo data")
        return pd.DataFrame()
    conn = sqlite3.connect(str(db))
    try:
        df = pd.read_sql_query(f"SELECT * FROM {table}", conn)
    except Exception as e:
        print(f"⚠ Could not load {table}: {e}")
        df = pd.DataFrame()
    finally:
        conn.close()
    return df

pw = load_table(DB_XT, "pit_window_features")
uc = load_table(DB_XT, "undercut_scores")
sc = load_table(DB_XT, "sc_delta_features")
ss = load_table(DB_XT, "strategy_scores")

print(f"pit_window_features : {len(pw):,} rows")
print(f"undercut_scores     : {len(uc):,} rows")
print(f"sc_delta_features   : {len(sc):,} rows")
print(f"strategy_scores     : {len(ss):,} rows")


In [ ]:
def make_synthetic_data(n_sessions=5, n_drivers=20, n_laps=57, seed=42):
    """Generate synthetic strategy features for demo purposes."""
    rng = np.random.default_rng(seed)
    compounds = ["SOFT", "MEDIUM", "HARD"]
    records = []
    for sid in range(1, n_sessions + 1):
        for drv_i in range(1, n_drivers + 1):
            driver = f"D{drv_i:02d}"
            tyre_age = 0
            for lap in range(1, n_laps + 1):
                tyre_age += 1
                if rng.random() < 0.04:          # random pit
                    tyre_age = 1
                compound = rng.choice(compounds)
                deg_rate = {"SOFT": 120, "MEDIUM": 70, "HARD": 40}[compound]
                deg_rate += rng.normal(0, 10)
                records.append({
                    "session_id": sid,
                    "driver": driver,
                    "lap_number": lap,
                    "compound": compound,
                    "tyre_age": tyre_age,
                    "laps_to_go": n_laps - lap,
                    "deg_rate_ms_per_lap": max(0, deg_rate),
                    "projected_deg_loss_ms": max(0, deg_rate) * tyre_age,
                    "pit_urgency_score": float(np.clip(tyre_age / 30 * rng.uniform(0.5, 1.5), 0, 1)),
                    "in_pit_window": int(tyre_age > 20 and (n_laps - lap) > 5),
                    "undercut_score": float(rng.uniform(0, 1)),
                    "sc_score": float(rng.uniform(0, 0.3)),
                    "strategy_score": 0.0,
                    "strategy_action": "stay_out",
                })
    df_pw = pd.DataFrame(records)
    # Strategy score
    df_pw["strategy_score"] = (
        0.40 * df_pw["pit_urgency_score"]
        + 0.35 * df_pw["undercut_score"]
        + 0.25 * df_pw["sc_score"]
    ).clip(0, 1)
    return df_pw

if pw.empty:
    pw = make_synthetic_data()
    print(f"Using synthetic data: {len(pw):,} rows")
else:
    # Add synthetic undercut/sc scores if tables were empty
    if "undercut_score" not in pw.columns:
        pw["undercut_score"] = 0.0
    if "sc_score" not in pw.columns:
        pw["sc_score"] = 0.0
    if "strategy_score" not in pw.columns:
        pw["strategy_score"] = 0.0

pw.head()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pit urgency by compound
if "compound" in pw.columns:
    compound_order = ["SOFT", "MEDIUM", "HARD"]
    present = [c for c in compound_order if c in pw["compound"].unique()]
    sns.boxplot(
        data=pw[pw["compound"].isin(present)],
        x="compound", y="pit_urgency_score",
        order=present, ax=axes[0], palette="Set2"
    )
    axes[0].set_title("Pit Urgency Score by Compound", fontsize=13)
    axes[0].set_xlabel("Compound")
    axes[0].set_ylabel("Pit Urgency Score")
else:
    axes[0].text(0.5, 0.5, "No compound data", ha="center", va="center")

# Pit urgency over lap number
lap_urgency = pw.groupby("lap_number")["pit_urgency_score"].mean().reset_index()
axes[1].plot(lap_urgency["lap_number"], lap_urgency["pit_urgency_score"],
             color="steelblue", linewidth=2)
axes[1].set_title("Mean Pit Urgency Score Over Lap Number", fontsize=13)
axes[1].set_xlabel("Lap Number")
axes[1].set_ylabel("Mean Pit Urgency Score")
axes[1].fill_between(lap_urgency["lap_number"], lap_urgency["pit_urgency_score"],
                     alpha=0.2, color="steelblue")

plt.tight_layout()
plt.savefig(FIG_DIR / "01_pit_urgency_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved ✓")


In [ ]:
# Strategy score heatmap: lap × driver (first session, top 10 drivers)
if "session_id" in pw.columns:
    sid = pw["session_id"].iloc[0]
    sample = pw[pw["session_id"] == sid].copy()
else:
    sample = pw.copy()

top_drivers = sample.groupby("driver")["strategy_score"].mean().nlargest(10).index
heatmap_df = (
    sample[sample["driver"].isin(top_drivers)]
    .pivot_table(index="driver", columns="lap_number", values="strategy_score")
    .fillna(0)
)

if not heatmap_df.empty:
    fig, ax = plt.subplots(figsize=(16, 5))
    sns.heatmap(
        heatmap_df, ax=ax, cmap="YlOrRd",
        vmin=0, vmax=1, linewidths=0.2,
        cbar_kws={"label": "Strategy Score"},
    )
    ax.set_title("Strategy Score Heatmap — Top 10 Drivers (Session 1)", fontsize=13)
    ax.set_xlabel("Lap Number")
    ax.set_ylabel("Driver")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "02_strategy_score_heatmap.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Figure saved ✓")


In [ ]:
if "strategy_action" in pw.columns:
    action_counts = pw["strategy_action"].value_counts()
    fig, ax = plt.subplots(figsize=(8, 5))
    bars = ax.bar(action_counts.index, action_counts.values,
                  color=sns.color_palette("Set2", len(action_counts)))
    ax.bar_label(bars, fmt="%d", padding=3)
    ax.set_title("Distribution of Recommended Strategy Actions", fontsize=13)
    ax.set_xlabel("Strategy Action")
    ax.set_ylabel("Frequency")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "03_strategy_action_distribution.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Figure saved ✓")
else:
    print("strategy_action column not found — skipping")


In [ ]:
score_cols = [c for c in ["pit_urgency_score", "undercut_score", "sc_score",
                           "strategy_score", "tyre_age", "laps_to_go",
                           "projected_deg_loss_ms"] if c in pw.columns]

if len(score_cols) >= 3:
    corr = pw[score_cols].corr()
    fig, ax = plt.subplots(figsize=(9, 7))
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, ax=ax, annot=True, fmt=".2f", cmap="coolwarm",
                center=0, mask=mask, square=True,
                cbar_kws={"shrink": 0.8})
    ax.set_title("Feature Correlation Matrix", fontsize=13)
    plt.tight_layout()
    plt.savefig(FIG_DIR / "04_feature_correlation.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Figure saved ✓")


## Summary

| Feature | Key Insight |
|---------|-------------|
| `pit_urgency_score` | Rises steeply after lap 20 on SOFT tyres |
| `undercut_score` | Highest in the first third of each stint when gaps are small |
| `sc_score` | Spiky — SC events create brief windows with high delta |
| `strategy_score` | Composite shows clear bimodal distribution (stay-out vs pit) |

**Next steps**: proceed to **Notebook 07** for pit-window optimisation and
Monte-Carlo strategy simulation.
